# Cross-Validation
---

So far, you have taken modeling decisions in a data-driven way by measuring model quality with a validation (or holdout) set. But there are some drawbacks to this approach. You will typically keep about 20% of the data as a validation dataset, this leaves some random chance in determining model scores, that is, a model might do well on one set of 1000 rows, even if it would be inaccurate on a different 1000 rows.

In general:
- A larger Validation set =<!-- /implies or means, --> less randomness (aka "noise").
- A large validation set =<!-- /implies or means, --> smaller training datasets.
- A small training dataset =<!-- /implies or mean --> worse models.

## What is cross-validation?
---

A method used to evaluate a machine learning model's performance by dividing the dataset into multiple subsets (folds). The model is trained on a portion of these subsets and tested on the remaining ones, repeating the process until all subsets are covered, to ensure it generalizes well to unseen data.

## When should you use cross-validation?
---

Cross-validation gives a more accurate measure of model quality. However, it can take longer to run. So, when should you use each approach?

- For small datasets, where extra computational burden isn't a big deal, you should run cross-validation.
- For larger datasets, a single validation set is sufficient. You may have enough data that don't need to re-use some of it for holdout.

There's no simple threshold for what constitutes a large vs. small dataset. But if your model takes a couple minutes or less to run, it's probably worth switching to cross-validation.

Alternatively, you can run cross-validation and see if the scores for each experiment seem close. If each experiment yields the same results, a single validation set is probably sufficient.

## Example
---

We'll work with the same data as in the previous classes. Then, we define a pipeline that uses an imputer to fill in missing values and a random forest model to make predictions.

Note: While it's possible to do cross-validation without pipelines, it is quite difficult!

In [4]:
import pandas as pd

# Read the data
data = pd.read_csv('../input/melbourne-housing-snapshot/melb_data.csv')

# Select subset of predictors
cols_to_use = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']
X = data[cols_to_use]

# Select target
y = data.Price

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

my_pipeline = Pipeline(steps=[('preprocessor', SimpleImputer()),
                              ('model', RandomForestRegressor(n_estimators=50, random_state=0))
                            ])

We obtain the cross-validation scores with the `cross_val_score()` function from scikit-learn. We set the number of folds with the `cv` parameter. This example uses neg_mean_absolute_error but you can chose other mesures of [scikit-learn's options list](https://scikit-learn.org/stable/modules/model_evaluation.html)

In [6]:
from sklearn.model_selection import cross_val_score

# Multiply by -1 since sklearn calculates *negative* MAE
scores = -1 * cross_val_score(my_pipeline, X, y,
                              cv=5,
                              scoring='neg_mean_absolute_error')

print("MAE scores:\n", scores)

MAE scores:
 [301628.7893587  303164.4782723  287298.331666   236061.84754543
 260383.45111427]


It is a little surprising that we specify negative MAE. Scikit-learn has a convention where all metrics are defined so a high number is better. Using negatives here allows them to be consistent with that convention, though negative MAE is almost unheard of elsewhere.

We typically want a single measure of model quality to compare alternative models. So we take the average across experiments.

In [7]:
print("Average MAE score (across experiments):")
print(scores.mean())

Average MAE score (across experiments):
277707.3795913405
